In [5]:
from pathlib import Path
import sys
import warnings

import pandas as pd
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
from IPython.display import Markdown, display

warnings.simplefilter("ignore")

project_root = Path.cwd().resolve().parents[1]
src_dir = project_root / "src" 
tex_dir = project_root / "results" / "tex" / "table" / "ols"
data_dir = project_root / "data"

sys.path.insert(0, str(src_dir))
from func_data_build import build_dataset

tex_dir.mkdir(parents=True, exist_ok=True)

data = build_dataset(data_dir)

y = data["pi_ppi"].dropna()
X = pd.DataFrame({
    "pi_prev": data["pi_ppi_prev"],
    "Epi": data["Epi_spf_cpi"],
    "x2": data["unemp_gap"],
    "oil": data["oil"],
    "Nhat": data["N_BN_cycle"],
})
df = pd.concat([y, X], axis=1).dropna()

periods = [
    ("1982-01-01", "2012-12-31"),
    ("1982-01-01", "1992-12-31"),
    ("1993-01-01", "2002-12-31"),
    ("2003-01-01", "2012-12-31"),
]


def fit_models(sub: pd.DataFrame):
    sub = sub.copy()
    sub["pi_prev_minus_Epi"] = sub["pi_prev"] - sub["Epi"]
    y = sub["pi_ppi"] - sub["Epi"]

    x0 = sm.add_constant(sub[["pi_prev_minus_Epi", "x2"]])
    m0 = sm.OLS(y, x0).fit()

    x1 = sm.add_constant(sub[["pi_prev_minus_Epi", "x2", "Nhat"]])
    m1 = sm.OLS(y, x1).fit()

    return m0, m1


def make_stata_table(models_dict: dict[str, sm.regression.linear_model.RegressionResultsWrapper]):
    info_dict = {
        "N": lambda x: f"{int(x.nobs)}",
        "R$^2$": lambda x: f"{x.rsquared:.3f}",
        "Adj. R$^2$": lambda x: f"{x.rsquared_adj:.3f}",
    }

    reg_order = ["const", "pi_prev_minus_Epi", "x2", "Nhat"]

    table = summary_col(
        list(models_dict.values()),
        stars=True,
        float_format="%.4f",
        model_names=list(models_dict.keys()),
        info_dict=info_dict,
        regressor_order=reg_order,
    )

    # 行名を見やすく
    table.tables[0].index = [
        idx.replace("const", "Constant")
           .replace("pi_prev_minus_Epi", r"$\alpha$")
           .replace("x2", r"$\kappa$")
           .replace("Nhat", r"$\gamma$")
        for idx in table.tables[0].index
    ]

    return table


# 全期間まとめた比較表
all_models = {}
for i, (start, end) in enumerate(periods, 1):
    sub = df.loc[start:end].dropna()
    m0, m1 = fit_models(sub)
    all_models[f"P{i} No $N_t^c$"] = m0
    all_models[f"P{i} With $N_t^c$"] = m1

stata_table_all = make_stata_table(all_models)

display(Markdown("## Stata-style OLS Table"))
display(stata_table_all)

latex_all = stata_table_all.as_latex()
(tex_dir / "ols_stata_style_all.tex").write_text(latex_all, encoding="utf-8")
print(f"Saved {tex_dir / 'ols_stata_style_all.tex'}")


# 各期間ごとの2列比較表
for i, (start, end) in enumerate(periods, 1):
    sub = df.loc[start:end].dropna()
    m0, m1 = fit_models(sub)

    period_models = {
        "Without $N_t^c$": m0,
        "With $N_t^c$": m1,
    }

    stata_table = make_stata_table(period_models)

    display(Markdown(f"## Period {i}: {start} to {end}"))
    display(stata_table)

    out_path = tex_dir / f"ols_period{i}_stata_style.tex"
    out_path.write_text(stata_table.as_latex(), encoding="utf-8")
    print(f"Saved {out_path}")

## Stata-style OLS Table

,P1 No $N_t^c$,P1 With $N_t^c$,P2 No $N_t^c$,P2 With $N_t^c$,P3 No $N_t^c$,P3 With $N_t^c$,P4 No $N_t^c$,P4 With $N_t^c$
Constant,-0.0914,-0.1631,-0.5624*,-0.5794*,-0.2558,-0.1796,1.0851,0.5554
,(0.2776),(0.3042),(0.3301),(0.3210),(0.3034),(0.3694),(1.0015),(1.5580)
$\alpha$,0.8028***,0.7975***,0.6674***,0.5165***,0.8212***,0.8140***,0.6921***,0.6937***
,(0.0539),(0.0548),(0.0987),(0.1262),(0.0957),(0.0988),(0.1205),(0.1219)
$\kappa$,0.0784,0.0618,0.3464**,0.5285***,0.2002,0.2821,0.2058,0.1426
,(0.1456),(0.1488),(0.1578),(0.1826),(0.3212),(0.3930),(0.3495),(0.3806)
$\gamma$,,-0.2835,,-1.1794*,,0.1945,,-1.5824
,,(0.4851),,(0.6410),,(0.5247),,(3.5376)
R-squared,0.6526,0.6535,0.6368,0.6652,0.6930,0.6941,0.5163,0.5190
R-squared Adj.,0.6468,0.6449,0.6191,0.6401,0.6764,0.6686,0.4902,0.4789


Saved /Users/satoshi/GitHub/FMI_NKPC_HSA_MCMC/output/tex/table/ols/ols_stata_style_all.tex


## Period 1: 1982-01-01 to 2012-12-31

,Without $N_t^c$,With $N_t^c$
Constant,-0.0914,-0.1631
,(0.2776),(0.3042)
$\alpha$,0.8028***,0.7975***
,(0.0539),(0.0548)
$\kappa$,0.0784,0.0618
,(0.1456),(0.1488)
$\gamma$,,-0.2835
,,(0.4851)
R-squared,0.6526,0.6535
R-squared Adj.,0.6468,0.6449


...

,Without $N_t^c$,With $N_t^c$
Constant,-0.2558,-0.1796
,(0.3034),(0.3694)
$\alpha$,0.8212***,0.8140***
,(0.0957),(0.0988)
$\kappa$,0.2002,0.2821
,(0.3212),(0.3930)
$\gamma$,,0.1945
,,(0.5247)
R-squared,0.6930,0.6941
R-squared Adj.,0.6764,0.6686


Saved /Users/satoshi/GitHub/FMI_NKPC_HSA_MCMC/output/tex/table/ols/ols_period3_stata_style.tex


## Period 4: 2003-01-01 to 2012-12-31

,Without $N_t^c$,With $N_t^c$
Constant,1.0851,0.5554
,(1.0015),(1.5580)
$\alpha$,0.6921***,0.6937***
,(0.1205),(0.1219)
$\kappa$,0.2058,0.1426
,(0.3495),(0.3806)
$\gamma$,,-1.5824
,,(3.5376)
R-squared,0.5163,0.5190
R-squared Adj.,0.4902,0.4789


Saved /Users/satoshi/GitHub/FMI_NKPC_HSA_MCMC/output/tex/table/ols/ols_period4_stata_style.tex
